In [1]:
import pandas as pd
import numpy as np
from AutoMLPreprocessor import AutoMLPreprocessor

from sklearn.model_selection import train_test_split

In [2]:
SAMPLE_SIZE = 0.01
X_train = pd.read_csv("data/train.csv")
X_test = pd.read_csv("data/test.csv")

y_train = X_train["Churn"]
X_train = X_train.drop(columns=["Churn"])


X_train, _, y_train, _ = train_test_split(
    X_train, 
    y_train, 
    train_size=SAMPLE_SIZE,
    stratify=y_train,
    random_state=42
)                                             

In [3]:
print(X_train.columns)
print(X_train.shape)

Index(['id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges'],
      dtype='object')
(5941, 20)


In [ ]:
preprocessor = AutoMLPreprocessor(
    target_col="Churn", 
    add_kmeans_features=True,
    feature_selection=True,
    add_poly_features=True,
    binnarise_features=False,
    remove_outliers=True,
    remove_multicollinearity=True,
    multicollinearity_threshold=0.95,
    id_threshold=0.95,
    random_state=42)

In [5]:
def create_features(df):
    df = df.copy()
    
    df['Contract_Internet'] = df['Contract'].astype(str) + '_' + df['InternetService'].astype(str)
    df['tenure_MonthlyCharges'] = df['tenure'] * df['MonthlyCharges']
    df['Senior_TechSupport'] = df['SeniorCitizen'].astype(str) + '_' + df['TechSupport'].astype(str)
    
    df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 6, 12, 24, 100], 
                                 labels=['0-6m', '6-12m', '12-24m', '24m+']).astype(str)
    df['MonthlyCharges_group'] = pd.cut(df['MonthlyCharges'], bins=[0, 35, 70, 100, 200], 
                                        labels=['low', 'medium', 'high', 'very_high']).astype(str)
    
    return df

X_train = create_features(X_train)
X_test = create_features(X_test)


X_train, y_train = preprocessor.fit_transform(X_train, y_train)
X_test = preprocessor.transform(X_test)

--- KMeans: Wytrenowano 15 klastrów ---
   -> Wybieranie top 7 cech do interakcji (ExtraTrees)...
   Wybrane kolumny do interakcji: ['FREQ_Contract_Internet', 'MonthlyCharges', 'Dist_Cluster_4', 'Dist_Cluster_10', 'Dist_Cluster_1', 'Dist_Cluster_6', 'FREQ_tenure']
   -> Wybieranie top 3 cech do interakcji (ExtraTrees)...
   Wybrane kolumny do powtórnej interakcji: ['SUB_FREQ_Contract_Internet_FREQ_tenure', 'SQR_FREQ_Contract_Internet', 'LOG_FREQ_Contract_Internet']

--- Selekcja Cech (Hybrid) ---
-> Etap 1: Wstępne usuwanie szumu (L1 + ExtraTrees)...
   -> Ocena Liniowa (L1 Logistic Regression)...
   -> Ocena Nieliniowa (ExtraTreesClassifier)...
   L1 wybrało: 50, Trees wybrało: 81
   Wspólna decyzja: Zachowano 104 z 184 cech (usunięto 80 najsłabszych).
-> Etap 2: Liczba cech (104) <= 250. Uruchamiam SFS Backward.


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:   11.2s
[Parallel(n_jobs=-1)]: Done 104 out of 104 | elapsed:   17.9s finished
Features: 103/1[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:    3.6s
[Parallel(n_jobs=-1)]: Done 103 out of 103 | elapsed:    8.4s finished
Features: 102/1[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:    3.3s
[Parallel(n_jobs=-1)]: Done 102 out of 102 | elapsed:    8.0s finished
Features: 101/1[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  42 tasks      | elapsed:    3.3s
[Parallel(n_jobs=-1)]: Done 101 out of 101 | elapsed:    7.8s finished
Features: 100/1[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done 

-> SFS zakończony. Wybrano 46 najlepszych cech.
--- Selekcja Cech Zakończona: Tryb = sfs ---

Ostateczne kolumny:['SUB_MonthlyCharges_Dist_Cluster_10', 'SUB_FREQ_Contract_Internet_Dist_Cluster_10', 'ADD_Dist_Cluster_4_Dist_Cluster_1', 'SUB_Dist_Cluster_10_Dist_Cluster_6', 'Dist_Cluster_5', 'MUL_MonthlyCharges_x_Dist_Cluster_6', 'Dist_Cluster_3', 'FREQ_Contract_Internet', 'DIV_Dist_Cluster_10_by_Dist_Cluster_4', 'ADD_Dist_Cluster_4_Dist_Cluster_6', 'DIV_MonthlyCharges_by_FREQ_tenure', 'TotalCharges', 'DIV_FREQ_Contract_Internet_by_MonthlyCharges', 'DIV_Dist_Cluster_1_by_Dist_Cluster_10', 'DIV_LOG_FREQ_Contract_Internet_by_SQR_FREQ_Contract_Internet', 'MUL_MonthlyCharges_x_Dist_Cluster_4', 'DIV_MonthlyCharges_by_FREQ_Contract_Internet', 'DIV_Dist_Cluster_10_by_Dist_Cluster_1', 'ADD_FREQ_Contract_Internet_Dist_Cluster_1', 'ADD_Dist_Cluster_6_FREQ_tenure', 'MonthlyCharges', 'DIV_Dist_Cluster_4_by_FREQ_Contract_Internet', 'MUL_MonthlyCharges_x_Dist_Cluster_10', 'DIV_Dist_Cluster_10_by_Dist_

In [11]:
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score
from xgboost import XGBClassifier

# 1. Pobranie nazw kolumn kategorycznych z preprocesora
cat_cols = preprocessor.get_categorical_cols(X_train)

# 2. Zmiana typu kolumn kategorycznych na 'category' w X_train (Wymóg XGBoosta)
# Upewniamy się, że modyfikujemy tylko te kolumny, które faktycznie istnieją w X_train
for col in cat_cols:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype('category')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['roc_auc', 'accuracy', 'balanced_accuracy', 'f1', 'precision', 'recall']
results_list = []

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    random_state=42,
    enable_categorical=True,
    n_jobs=-1
)

model.fit(X_train, y_train)

cv_results = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

results_list.append({
    "Model": "XGBoost",
    "ROC AUC": np.mean(cv_results['test_roc_auc']),
    "Accuracy": np.mean(cv_results['test_accuracy']),
    "Balanced Acc": np.mean(cv_results['test_balanced_accuracy']),
    "F1 Score": np.mean(cv_results['test_f1']),
    "Precision": np.mean(cv_results['test_precision']),
    "Recall": np.mean(cv_results['test_recall'])
})

results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values(by="ROC AUC", ascending=False).reset_index(drop=True)

print("\n" + "-"*80)
print("Wyniki po optymalizacji hiperparametrów:")
print(results_df.to_string(index=False))


--------------------------------------------------------------------------------
Wyniki po optymalizacji hiperparametrów:
  Model  ROC AUC  Accuracy  Balanced Acc  F1 Score  Precision  Recall
XGBoost 0.899439  0.846323      0.762996  0.641879    0.67658 0.61137


In [ ]:
aimport optuna
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_score
from xgboost import XGBClassifier
import pandas as pd
import numpy as np

# 1. Pobranie nazw kolumn kategorycznych z preprocesora
cat_cols = preprocessor.get_categorical_cols(X_train)

# 2. Zmiana typu kolumn kategorycznych na 'category' w X_train (Wymóg XGBoosta)
# Upewniamy się, że modyfikujemy tylko te kolumny, które faktycznie istnieją w X_train
for col in cat_cols:
    if col in X_train.columns:
        X_train[col] = X_train[col].astype('category')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['roc_auc', 'accuracy', 'balanced_accuracy', 'f1', 'precision', 'recall']
results_list = []

# Liczba prób
N_TRIALS = 100

optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f"Rozpoczynam strojenie hiperparametrów dla XGBoost ({N_TRIALS} prób)...")
print("-" * 60)

def objective(trial):
    # Znacznie rozszerzona i zoptymalizowana pod XGBoost przestrzeń poszukiwań
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        
        # Konfiguracja pod zmienne kategoryczne
        'enable_categorical': True,
        'tree_method': 'hist', 
        
        'random_state': 42,
        'n_jobs': -1 # Wykorzystanie wszystkich rdzeni
    }
    
    model = XGBClassifier(**params)
        
    score = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1)
    return score.mean()

study = optuna.create_study(direction='maximize')
# show_progress_bar=True wymaga doinstalowania biblioteki tqdm (pip install tqdm)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = study.best_params
print(f"\nNajlepsze znalezione parametry: \n{best_params}")

# Dodanie parametrów stałych do najlepszych parametrów zwróconych przez Optunę
best_params['enable_categorical'] = True
best_params['tree_method'] = 'hist'
best_params['random_state'] = 42
best_params['n_jobs'] = -1

# Inicjalizacja najlepszego modelu
best_model = XGBClassifier(**best_params)

# Finalna walidacja krzyżowa ze wszystkimi metrykami
print("\nPrzeprowadzam finalną ewaluację najlepszego modelu...")
cv_results = cross_validate(best_model, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

results_list.append({
    "Model": "XGBoost",
    "ROC AUC": np.mean(cv_results['test_roc_auc']),
    "Accuracy": np.mean(cv_results['test_accuracy']),
    "Balanced Acc": np.mean(cv_results['test_balanced_accuracy']),
    "F1 Score": np.mean(cv_results['test_f1']),
    "Precision": np.mean(cv_results['test_precision']),
    "Recall": np.mean(cv_results['test_recall'])
})

results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values(by="ROC AUC", ascending=False).reset_index(drop=True)

print("\n" + "-"*80)
print("Wyniki po optymalizacji hiperparametrów:")
print(results_df.to_string(index=False))

Rozpoczynam strojenie hiperparametrów dla XGBoost (100 prób)...
------------------------------------------------------------


  0%|          | 0/100 [00:00<?, ?it/s]


Najlepsze znalezione parametry: 
{'n_estimators': 355, 'learning_rate': 0.010250921521472054, 'max_depth': 6, 'subsample': 0.6914252576677119, 'colsample_bytree': 0.9745303930027814, 'min_child_weight': 6, 'gamma': 0.49113565630537703}

Przeprowadzam finalną ewaluację najlepszego modelu...

--------------------------------------------------------------------------------
Wyniki po optymalizacji hiperparametrów:
  Model  ROC AUC  Accuracy  Balanced Acc  F1 Score  Precision   Recall
XGBoost 0.899018  0.852384      0.772207  0.656423   0.689919 0.626315
